# Multimodal AI Pipeline: Text-to-Image Verification
**Objective:** Demonstrate the integration of a fine-tuned NER Transformer model (NLP) and a ResNet-18 Image Classifier (CV) to verify if the text description matches the visual content.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML

from src.classifier.inference import load_model as load_cv_model
from src.classifier.inference import predict as predict_cv
from src.ner.inference import extract_animals

import warnings
warnings.filterwarnings('ignore')

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CV_MODEL_PATH = "../models/best_animal_classifier.pth"
NER_MODEL_PATH = "../models/ner_animal/best_model"

print(f"Loading models into memory on {DEVICE}...")

cv_model = load_cv_model(CV_MODEL_PATH, num_classes=10, device=DEVICE)

print("Models successfully loaded and ready for inference.")

In [ ]:
def evaluate_and_display(text: str, image_path: str):
    """
    Executes the pipeline, extracts entities, predicts the image class, 
    and renders a formatted visualization of the results.
    """
    # 1. NLP Inference
    extracted_animals = extract_animals(text, NER_MODEL_PATH)
    extracted_animals_lower = [a.lower() for a in extracted_animals]

    # 2. CV Inference
    predicted_animal = predict_cv(image_path, cv_model, DEVICE)
    predicted_animal_lower = predicted_animal.lower()

    # 3. Logic & Matching
    is_match = predicted_animal_lower in extracted_animals_lower

    fig, ax = plt.subplots(figsize=(4, 4))
    try:
        img = Image.open(image_path)
        ax.imshow(img)
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
    ax.axis('off')
    plt.show()

    match_color = "green" if is_match else "red"
    match_text = "MATCH (TRUE)" if is_match else "MISMATCH (FALSE)"
    
    html_out = f"""
    <div style="font-family: Arial, sans-serif; padding: 10px; border: 1px solid #ddd; border-radius: 5px; width: 600px;">
        <p><b> Input Text:</b> <i>"{text}"</i></p>
        <hr style="border: 0.5px solid #eee;">
        <p><b>NER Extraction:</b> {extracted_animals if extracted_animals else 'None'}</p>
        <p><b>CV Prediction:</b> {predicted_animal.capitalize()}</p>
        <p style="font-size: 16px;"><b>Result:</b> <span style="color: {match_color}; font-weight: bold;">{match_text}</span></p>
    </div>
    """
    display(HTML(html_out))

## 1. Standard Use Cases
Testing the baseline functionality where the text is straightforward and matches (or clearly contradicts) the image.

In [ ]:
# Test Case 1.1: Perfect Match
image_path_cow = "../data/Animals-10/cow/1.jpg" 
text_match = "There is a large brown cow grazing in the picture."

evaluate_and_display(text_match, image_path_cow)

In [ ]:
# Test Case 1.2: Clear Mismatch
text_mismatch = "Look at this cute little cat sleeping."

evaluate_and_display(text_mismatch, image_path_cow)

## 2. Edge Cases
Testing system robustness against complex sentences, missing entities, and multiple targets.

In [ ]:
# Edge Case 2.1: Multiple animals mentioned in the text (Logical OR validation)
# The image is a cow, but the text mentions both a horse and a cow. The pipeline should return True.
text_multiple = "I can't tell if this is a horse or a cow, but it's huge."

evaluate_and_display(text_multiple, image_path_cow)

In [ ]:
# Edge Case 2.2: Zero entities in text (False Positive Prevention)
# The image contains an animal, but the user text describes the background.
text_empty = "This is a beautiful green landscape with a clear blue sky."

evaluate_and_display(text_empty, image_path_cow)

In [ ]:
# Edge Case 2.2: Zero entities in text (False Positive Prevention)
# The image contains an animal, but the user text describes the background.
text_empty = "This is a beautiful green landscape with a clear blue sky."

evaluate_and_display(text_empty, image_path_cow)